# VoiceContext Agent — How it works & interactive demo

This notebook explains the agent architecture **step by step** and lets you run each piece individually.

**No audio hardware required** — we start in text-only mode so you can understand the loop before adding NeMo.

---
## The agent loop (4 steps)

```
Audio/Text
   ↓
[1. PERCEIVE]  VoiceContext → TurnContext (text + emotion + paralinguistics + intent)
   ↓
[2. THINK]     LLM receives TurnContext + history + system prompt → produces reasoning
   ↓
[3. DECIDE]    LLM chooses: call a tool OR generate text reply
   ↓
[4. ACT]       Execute tool / speak reply → append to history → loop back to step 1
```

The key insight: **VoiceContext collapses steps 1-2 into a single MCP tool call** (`get_turn_context`). The LLM never touches raw audio — it only sees the structured object.

---
## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, "../apps/demo/src")   # simple_agent.py lives in src/

from dotenv import load_dotenv
load_dotenv("../.env")

if not os.environ.get("ANTHROPIC_API_KEY"):
    print("⚠️  ANTHROPIC_API_KEY not set — add it to your .env file")
else:
    print("✓ API key found")

---
## 1. Perceive — text → TurnContext

This is what VoiceContext does for you. We use a stub here; the real engine uses NeMo + SpeechBrain.

In [ ]:
from simple_agent import perceive_from_text
import json

# Try different sentences and observe how the TurnContext changes
test_sentences = [
    "I'm really frustrated, nothing is working and I've been waiting for an hour!",
    "Uh, hi, um, I was just wondering... how do I reset my password?",
    "Thanks, everything's working great now!",
    "I hate this service, I want a refund immediately.",
]

for sentence in test_sentences:
    ctx = perceive_from_text(sentence)
    print(f"Input  : {sentence[:60]}..." if len(sentence) > 60 else f"Input  : {sentence}")
    print(f"Emotion: {ctx.emotion.dominant} (valence={ctx.emotion.valence:+.2f}, arousal={ctx.emotion.arousal:.2f})")
    print(f"Intent : {ctx.intent.name} (conf={ctx.intent.confidence:.2f})")
    print(f"Hesit. : {ctx.paralinguistics.hesitations}")
    print()

print("─" * 50)
print("\nFull TurnContext object:")
ctx = perceive_from_text("I'm really frustrated, nothing is working!")
print(ctx.to_prompt_str())

---
## 2. Think — how the LLM sees TurnContext

The LLM receives the TurnContext as a formatted user message. Here's exactly what it looks like:

In [ ]:
from simple_agent import SYSTEM_PROMPT, perceive_from_text

ctx = perceive_from_text("I've been waiting for 3 days and nobody responded to my ticket!")

print("=== SYSTEM PROMPT (what the agent is) ===")
print(SYSTEM_PROMPT)

print("\n=== USER MESSAGE (what the LLM receives per turn) ===")
user_msg = f"[Turn 1 — {ctx.turn_id}]\n{ctx.to_prompt_str()}"
print(user_msg)

---
## 3. Decide — tools the agent can call

In [ ]:
from simple_agent import TOOLS
import json

print(f"Agent has {len(TOOLS)} tools:\n")
for tool in TOOLS:
    print(f"  • {tool['name']}")
    print(f"    {tool['description']}")
    props = tool['input_schema'].get('properties', {})
    print(f"    Args: {', '.join(props.keys())}")
    print()

---
## 4. One full agent loop — step by step

Let's trace a complete turn manually to understand exactly what happens.

In [ ]:
import anthropic
import json
from simple_agent import SYSTEM_PROMPT, TOOLS, execute_tool, perceive_from_text

client = anthropic.Anthropic()

# ─── Step 1: Perceive ─────────────────────────────────────────────
user_speech = "I'm so angry, your service is completely broken and I need help NOW!"
ctx = perceive_from_text(user_speech)

print("[1] PERCEIVE")
print(f"  Text    : {ctx.transcription.text}")
print(f"  Emotion : {ctx.emotion.dominant} (valence={ctx.emotion.valence:+.2f})")
print(f"  Intent  : {ctx.intent.name}")

# ─── Step 2: Build message for LLM ───────────────────────────────
history = [
    {"role": "user", "content": f"[Turn 1 — {ctx.turn_id}]\n{ctx.to_prompt_str()}"}
]

print("\n[2] THINK — calling LLM...")
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=512,
    system=SYSTEM_PROMPT,
    tools=TOOLS,
    messages=history,
)

print(f"  Stop reason : {response.stop_reason}")
print(f"  Content blocks: {[b.type for b in response.content]}")

# ─── Step 3: Decide ───────────────────────────────────────────────
print("\n[3] DECIDE")
if response.stop_reason == "tool_use":
    for block in response.content:
        if block.type == "tool_use":
            print(f"  → Tool call: {block.name}")
            print(f"    Input: {json.dumps(block.input, indent=6)}")
else:
    for block in response.content:
        if hasattr(block, "text"):
            print(f"  → Text reply: {block.text}")

# ─── Step 4: Act ──────────────────────────────────────────────────
print("\n[4] ACT")
if response.stop_reason == "tool_use":
    tool_results = []
    for block in response.content:
        if block.type == "tool_use":
            result = execute_tool(block.name, block.input)
            print(f"  Tool result: {json.dumps(result, indent=4)}")
            tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(result)})

    # LLM gets tool result and generates final reply
    history.append({"role": "assistant", "content": response.content})
    history.append({"role": "user", "content": tool_results})
    final_response = client.messages.create(
        model="claude-sonnet-4-6", max_tokens=256,
        system=SYSTEM_PROMPT, tools=TOOLS, messages=history
    )
    for block in final_response.content:
        if hasattr(block, "text"):
            print(f"\n  Final reply: {block.text}")

print(f"\n  History now has {len(history)+1} messages")
print("  → Loop back to step 1 for next turn")

---
## 5. Full agent — multi-turn conversation

Now let's run multiple turns and see how memory accumulates.

In [ ]:
from simple_agent import VoiceAgent, perceive_from_text

agent = VoiceAgent()

# Simulate a multi-turn support conversation
conversation = [
    "Hi, I need help with my account.",
    "I can't log in, I keep getting an error.",
    "Ugh, I've been trying for 30 minutes, this is so frustrating!",
    "Finally! Can you send me the reset link by email?",
]

print("=" * 60)
print("  Multi-turn voice agent conversation")
print("=" * 60)

for i, utterance in enumerate(conversation, 1):
    print(f"\n[Turn {i}]")
    print(f"🎤  User   : {utterance}")

    ctx = perceive_from_text(utterance)
    print(f"   Context: {ctx.emotion.dominant} (valence={ctx.emotion.valence:+.2f}), intent={ctx.intent.name}")

    response = agent.process_turn(ctx)
    print(f"🤖  Agent  : {response}")

print(f"\nHistory size: {len(agent.history)} messages ({agent.turn_count} turns)")

---
## 6. The power of emotion context — A/B test

Same words, different emotional context → different agent behavior.

In [ ]:
from simple_agent import VoiceAgent, TurnContext, Transcription, EmotionResult, ParalinguisticFeatures, IntentResult
import uuid
from datetime import datetime, timezone

def make_ctx(text, emotion, valence, arousal, hesitations=0):
    return TurnContext(
        turn_id=str(uuid.uuid4())[:8],
        timestamp=datetime.now(timezone.utc).isoformat(),
        transcription=Transcription(text=text),
        emotion=EmotionResult(dominant=emotion, valence=valence, arousal=arousal),
        paralinguistics=ParalinguisticFeatures(hesitations=hesitations),
        intent=IntentResult(name="inquiry", confidence=0.8),
    )

SAME_TEXT = "I need help with my account."
variants = [
    ("Calm, neutral",    make_ctx(SAME_TEXT, "neutral",  0.0,  0.1, hesitations=0)),
    ("Confused, hesitant",make_ctx(SAME_TEXT, "neutral",  0.0,  0.2, hesitations=4)),
    ("Angry, distressed",make_ctx(SAME_TEXT, "anger",   -0.75, 0.8, hesitations=0)),
    ("Happy, energetic", make_ctx(SAME_TEXT, "joy",      0.7,  0.6, hesitations=0)),
]

print(f"Text: '{SAME_TEXT}'  — same words, 4 different emotional contexts")
print("=" * 60)

for label, ctx in variants:
    agent = VoiceAgent()  # fresh agent each time
    response = agent.process_turn(ctx)
    print(f"\n[{label}]")
    print(f"  → {response}")

---
## 7. NeMo Cache-Aware Streaming STT

`nvidia/nemotron-speech-streaming-en-0.6b` is a FastConformer-RNNT model trained for **chunk-based streaming**. The key idea: instead of seeing the whole audio at once, it processes fixed-size chunks (e.g. 160 ms) and keeps an **attention cache** across chunks so context is never lost.

```
audio ──► [chunk 0]──cache──► [chunk 1]──cache──► [chunk 2]──cache──► …
                ↓                   ↓                   ↓
           partial hyp         partial hyp         final hyp
```

We run three tests:
1. **Model info** — verify the model loads and inspect its config
2. **Streaming on a WAV file** — chunk-by-chunk processing, print partial hypotheses
3. **Compare vs offline** — correctness check: streaming final == offline output

In [ ]:
import os, time
import numpy as np
import torch
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# ── Prerequisites check ────────────────────────────────────────────────────
try:
    import nemo.collections.asr as nemo_asr
    from nemo.collections.asr.parts.utils.streaming_utils import CacheAwareStreamingAudioBuffer
    import torchaudio
    NEMO_OK = True
except ImportError as e:
    NEMO_OK = False
    print(f"NeMo not available: {e}")
    print("\nInstall (heavy — ~3 GB):")
    print("  uv add --group dev 'nemo_toolkit[asr]'")
    print("  # then re-lock: uv lock && docker compose build notebook")

if not NEMO_OK:
    raise SystemExit("Install NeMo and re-run this cell")

# ── Config ─────────────────────────────────────────────────────────────────
MODEL_NAME  = "nvidia/nemotron-speech-streaming-en-0.6b"
SAMPLE_RATE = 16_000
CHUNK_MS    = 160   # supported: 80 | 160 | 560 | 1120 ms
CHUNK_SAMPS = int(SAMPLE_RATE * CHUNK_MS / 1000)

device = torch.device(
    "mps"  if torch.backends.mps.is_available()  else
    "cuda" if torch.cuda.is_available()           else "cpu"
)

# ── 1. Load model ──────────────────────────────────────────────────────────
print(f"Loading {MODEL_NAME}  (first run: ~600 MB download, cached after)")
t0 = time.perf_counter()
model = nemo_asr.models.EncDecRNNTBPEModel.from_pretrained(MODEL_NAME).to(device).eval()
print(f"  Loaded in {time.perf_counter()-t0:.1f}s  |  device={device}")
print(f"  Params   : {sum(p.numel() for p in model.parameters())/1e6:.0f} M")
print(f"  Sample rate: {model.cfg.preprocessor.sample_rate} Hz")
print(f"  Chunk size : {CHUNK_MS} ms = {CHUNK_SAMPS} samples")

# ── 2. Set up CacheAwareStreamingAudioBuffer ───────────────────────────────
streaming_buf = CacheAwareStreamingAudioBuffer(model=model, online_normalization=True)
cache = streaming_buf.get_buffers_state()

# Warmup: one silent chunk — initialises all cache tensors
with torch.no_grad():
    streaming_buf.process_chunk(np.zeros(CHUNK_SAMPS, dtype=np.float32), *cache)
cache = streaming_buf.get_buffers_state()
print("\nCache initialised (warmup done)")
print(f"  Cache state: {len(cache)} tensors, "
      f"shapes = {[tuple(c.shape) for c in cache if hasattr(c, 'shape')][:3]} ...")

In [ ]:
# ── 5. Feed NeMo transcript → VoiceContext agent ───────────────────────────
if os.path.exists(AUDIO_FILE) and streaming_final:
    from simple_agent import VoiceAgent, perceive_from_text

    # In production: replace perceive_from_text with a real SpeechBrain call
    # that uses the audio file to extract emotion + paralinguistics.
    ctx = perceive_from_text(streaming_final)
    agent = VoiceAgent()
    reply = agent.process_turn(ctx)

    print("NeMo transcript → TurnContext → LLM agent")
    print(f"\n  Transcript : {streaming_final!r}")
    print(f"  Emotion    : {ctx.emotion.dominant} (valence={ctx.emotion.valence:+.2f})")
    print(f"  Intent     : {ctx.intent.name}")
    print(f"\n  Agent reply: {reply}")
else:
    print("Run the streaming cell above first (needs AUDIO_FILE).")

In [ ]:
from simple_agent import perceive_from_text

PROBES = [
    ("anger / complaint",     "This is terrible! I've been waiting hours and nothing works!"),
    ("joy / statement",       "Everything works perfectly, thank you so much!"),
    ("uncertain / inquiry",   "Uh, hi, um, I was wondering... how do I reset my password?"),
    ("neutral / inquiry",     "How do I reset my password?"),
    ("neutral / statement",   "I received the confirmation email."),
    ("mixed",                 "I love the product but the checkout is completely broken."),
]

print(f"{'Label':<26} {'Emotion':<10} {'Valence':>8} {'Arousal':>8} {'Intent':<12} {'Hesit':>5}")
print("─" * 74)
for label, text in PROBES:
    ctx = perceive_from_text(text)
    print(f"{label:<26} {ctx.emotion.dominant:<10} {ctx.emotion.valence:>+8.2f} "
          f"{ctx.emotion.arousal:>8.2f} {ctx.intent.name:<12} {ctx.paralinguistics.hesitations:>5}")

print("""
─── Swap in real SpeechBrain emotion ────────────────────────────────────────
from speechbrain.inference.classifiers import EncoderClassifier
classifier = EncoderClassifier.from_hparams(
    source="speechbrain/emotion-recognition-wav2vec2-IEMOCAP"
)
# out = classifier.classify_file(audio_path)
# label, score = out[3][0], float(out[1][0])
# valence = {"happiness":0.7, "neutral":0.0, "sadness":-0.4, "anger":-0.8}.get(label, 0)
# EmotionResult(dominant=label, valence=valence, confidence=score)
""")

---
## 10. Emotion & Intent — deep probe

Stress-test the perception layer across emotion / intent combinations. Once real audio is wired in, replace `perceive_from_text` with the SpeechBrain path shown at the bottom.

In [ ]:
import time, anthropic
from simple_agent import SYSTEM_PROMPT, TOOLS, perceive_from_text

client = anthropic.Anthropic()

turns = [
    "Hi, I need help with my account.",
    "I can't log in, I keep getting an error.",
    "I'm really frustrated, nothing is working!",
]

print("Turn-by-turn prompt-caching stats\n")
print(f"{'Turn':<6} {'Latency':>10} {'Input':>8} {'Cached':>8} {'Output':>8}")
print("─" * 46)

history = []
for i, text in enumerate(turns, 1):
    ctx = perceive_from_text(text)
    history.append({"role": "user", "content": f"[Turn {i}]\n{ctx.to_prompt_str()}"})

    t0 = time.perf_counter()
    resp = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=256,
        system=[{"type": "text", "text": SYSTEM_PROMPT,
                 "cache_control": {"type": "ephemeral"}}],
        tools=TOOLS,
        messages=history,
    )
    latency_ms = (time.perf_counter() - t0) * 1000
    cached = getattr(resp.usage, "cache_read_input_tokens", 0)
    print(f"{i:<6} {latency_ms:>9.0f}ms {resp.usage.input_tokens:>8} {cached:>8} {resp.usage.output_tokens:>8}")
    history.append({"role": "assistant", "content": "ok"})

print("\n  'Cached' rises from turn 2+ — system prompt tokens read from cache")
print("  Cache TTL: 5 min.  Cost: 10% of normal input price.")

---
## 9. Claude Prompt Caching — cut cost on the system prompt

The system prompt is fixed across every turn. Adding `cache_control` tells Claude to store it server-side: subsequent turns read from cache at **~10 % of normal input token cost**.

> Note: this is *Claude-side* caching — different from the NeMo *attention cache* above.

In [ ]:
# ── 4. Compare streaming vs offline (correctness check) ───────────────────
# From: examples/asr/asr_cache_aware_streaming/speech_to_text_cache_aware_streaming_infer.py
# Streaming output should match offline output on the same file.

if os.path.exists(AUDIO_FILE):
    print("Offline transcription (no streaming, full file at once)...")
    with torch.no_grad():
        offline_result = model.transcribe([AUDIO_FILE])
    offline_text = (offline_result[0].text
                    if hasattr(offline_result[0], "text")
                    else str(offline_result[0]))

    print(f"\n  Streaming : {streaming_final!r}")
    print(f"  Offline   : {offline_text!r}")

    match = streaming_final.strip().lower() == offline_text.strip().lower()
    if match:
        print("\n  ✓ Exact match — cache-aware streaming is working correctly")
    else:
        # Minor differences are expected at utterance boundaries
        from difflib import SequenceMatcher
        ratio = SequenceMatcher(None, streaming_final, offline_text).ratio()
        print(f"\n  ≈ Similarity: {ratio:.0%}  (small diffs at chunk boundaries are normal)")
else:
    print("Add an audio file to run the compare-vs-offline test.")

In [ ]:
# ── 3. Streaming on a WAV file ─────────────────────────────────────────────
# Put any 16 kHz mono WAV at this path, or record one with:
#   sox -d -r 16000 -c 1 ../apps/demo/test_audio/sample.wav trim 0 5
AUDIO_FILE = "../apps/demo/test_audio/sample.wav"

if not os.path.exists(AUDIO_FILE):
    print(f"No file at {AUDIO_FILE}")
    print("  → record one, or point AUDIO_FILE at any 16kHz mono WAV")
else:
    audio, sr = torchaudio.load(AUDIO_FILE)
    if sr != SAMPLE_RATE:
        audio = torchaudio.functional.resample(audio, sr, SAMPLE_RATE)
    audio_np = audio.mean(0).numpy().astype(np.float32)   # mono float32

    # Fresh buffer + cache for this file
    sbuf = CacheAwareStreamingAudioBuffer(model=model, online_normalization=True)
    cache2 = sbuf.get_buffers_state()

    print(f"File     : {AUDIO_FILE}")
    print(f"Duration : {len(audio_np)/SAMPLE_RATE:.2f}s  ({len(audio_np)} samples)")
    print(f"Chunks   : {len(audio_np)//CHUNK_SAMPS} × {CHUNK_MS}ms\n")

    streaming_final = ""
    t0 = time.perf_counter()

    for i in range(0, len(audio_np), CHUNK_SAMPS):
        chunk = audio_np[i : i + CHUNK_SAMPS]
        if len(chunk) < CHUNK_SAMPS:                       # pad last chunk
            chunk = np.pad(chunk, (0, CHUNK_SAMPS - len(chunk)))

        with torch.no_grad():
            partial, *cache2 = sbuf.process_chunk(chunk, *cache2)

        chunk_idx = i // CHUNK_SAMPS
        ts = chunk_idx * CHUNK_MS / 1000
        if partial:
            print(f"  [{ts:5.2f}s]  {partial!r}")
            streaming_final = partial

    rtf = (time.perf_counter() - t0) / (len(audio_np) / SAMPLE_RATE)
    print(f"\nStreaming final : {streaming_final!r}")
    print(f"RTF             : {rtf:.2f}x  (< 1.0 = faster than real-time)")

---
## 8. Run the full interactive demo

Open a terminal and run:

```bash
conda activate voicecontext
cd voicecontext

# Text mode (no audio needed)
python apps/demo/simple_agent.py --text

# Live mic (requires NeMo + pyaudio)
export PYTORCH_ENABLE_MPS_FALLBACK=1
python apps/demo/simple_agent.py --mic

# Audio file
python apps/demo/simple_agent.py --audio scripts/test_audio/nemo_sample.wav
```

## What's next

1. **Verify NeMo works** → run `notebooks/01_nemo_fastconformer_streaming.ipynb`
2. **Add real emotion** → integrate SpeechBrain into `perceive_from_audio()`
3. **Package as MCP** → `packages/mcp-server` — agents call `get_turn_context` via MCP protocol
4. **Add TTS** → response text → speech synthesis → speaker
5. **Deploy** → `docker/docker-compose.yml` — expose MCP over HTTP/SSE